In [1]:
import pandas as pd

In [2]:
def assign_destination(destination: str) -> int:
    if destination == "Broadcast": 
        return 1
    else:
        return 0


def check_null_value(string_to_check: str) -> str:
    if pd.isna(string_to_check):
        return ""
    else:
        return string_to_check


def assign_device(device: str, source: str) -> str:
    if device in ["Phone", "Phone Scan", "Phone Google"]:
        return "Phone"
    else:
        return device


def encode_company_id(company: str, source: str, service_data: str) -> str:
    # Defined, but not named company identifiers f.e. 0x34f5
    if str(company).startswith("0x") and len(company) == 6:
        return "Unknown"
    # Assign the Galaxy S22 "Anonymous" source the company Samsung
    elif pd.isna(company) and source == "Anonymous":
        return "Samsung Electronics Co. Ltd."
    # Reason?
    elif pd.isna(company) and str(service_data).startswith("4a17235"):
        return "Samsung Electronics Co. Ltd."
    elif pd.isna(company):
        return "Undefined"
    else:
        return company


def extract_time(delta_string: str) -> int:
    if pd.isna(delta_string):
        return 0
    else:
        s = str(delta_string)
        num = "".join(ch for ch in s if ch.isdigit())
        return int(num) if num else 0


def check_company_id_existence(company_id: str) -> int:
    if company_id == "Undefined":
        return 0
    else:
        return 1


def extract_length(entry_to_exctract: str) -> int:
    if pd.isna(entry_to_exctract):
        return 0
    else:
        return len(entry_to_exctract)


def is_adv_channel(channel: int) -> bool:
    if channel in [37, 38, 39]:
        return True
    else:
        return False


def check_entry_existence(entry_to_check: str) -> int:
    if pd.isna(entry_to_check):
        return 0
    else:
        return 1


In [3]:
# Read the input CSV file and rename some columns - robust to both naming schemes (Channel vs Channel_Index, UUID16 vs UUID_16)
def read_data_to_df(file_name: str) -> pd.DataFrame:
    dataset_df = pd.read_csv(file_name, encoding='ISO-8859-1', low_memory=False)
    dataset_df.columns = dataset_df.columns.str.replace("Packet time (start to end)", "packet_start_end")
    dataset_df.columns = dataset_df.columns.str.replace("Delta time (end to start)", "delta_end_start")
    dataset_df.columns = dataset_df.columns.str.replace("Delta time (start to start)", "delta_start_start")
    dataset_df.columns = dataset_df.columns.str.replace(" ", "_")
    dataset_df.columns = dataset_df.columns.str.replace("Test", "Device")
    # Normalize column variants for compatibility
    # Channel_Index vs Channel
    if "Channel" in dataset_df.columns and "Channel_Index" not in dataset_df.columns:
        dataset_df = dataset_df.rename(columns={"Channel": "Channel_Index"})
    # UUID_16 vs UUID16
    if "UUID16" in dataset_df.columns and "UUID_16" not in dataset_df.columns:
        dataset_df = dataset_df.rename(columns={"UUID16": "UUID_16"})
    # Type vs AD_Type already maps to Type after space replace? Keep both
    if "AD_Type" in dataset_df.columns and "Type" not in dataset_df.columns:
        dataset_df = dataset_df.rename(columns={"AD_Type": "Type"})
    # Ensure expected columns exist
    return dataset_df

# --- Multi-Group Whitelist (ground truth from controlled inventory) ---
# Each physical device has 1 AdvA (Buds/AirTag) or 1 ScanA per rotation window (Phone)
# Phones rotate RPA every ~400s => 3-4 ScanA per 20min capture
PHONE_G1 = ["71:9c:05:bf:d0:e9","59:2a:92:26:ae:ac","48:1a:0a:8d:24:91"]
PHONE_G2 = ["62:ad:82:a0:76:e6","7b:1f:3c:ec:d0:6c","65:cd:d7:af:54:cb","49:f7:63:04:de:64"]
BUDS_G1 = ["dc:b9:92:eb:f6:5d"]
BUDS_G2 = ["f0:c8:c1:88:8c:53"]
AIRTAG_G1 = ["fb:28:4e:b6:e1:c1"]
AIRTAG_G2 = ["d2:5f:07:c2:32:c0"]

# Reverse map: Source -> (labelled_device, sublabel_device, group_id)
SOURCE_TO_GROUP = {}
for s in PHONE_G1: SOURCE_TO_GROUP[s] = ("Phone", "Phone Scan", 1)
for s in PHONE_G2: SOURCE_TO_GROUP[s] = ("Phone", "Phone Scan", 2)
for s in BUDS_G1: SOURCE_TO_GROUP[s] = ("Buds", "Buds", 1)
for s in BUDS_G2: SOURCE_TO_GROUP[s] = ("Buds", "Buds", 2)
for s in AIRTAG_G1: SOURCE_TO_GROUP[s] = ("AirTag", "AirTag", 1)
for s in AIRTAG_G2: SOURCE_TO_GROUP[s] = ("AirTag", "AirTag", 2)

# Also keep device-type only sets for fallback heuristic
ALL_BUDS = set(BUDS_G1 + BUDS_G2)
ALL_PHONES_SCAN = set(PHONE_G1 + PHONE_G2)
ALL_AIRTAGS = set(AIRTAG_G1 + AIRTAG_G2)

def initialize_source_dictionaries(all_unique_sources: list) -> None:
    for source in all_unique_sources:
        if str(source) != "nan":
            source_dictionaries[source] = {}
            source_dictionaries[source]["count"] = 0
            source_dictionaries[source]["malformed_count"] = 0
            source_dictionaries[source]["highest_rssi"] = 0
            source_dictionaries[source]["lowest_rssi"] = -100
            source_dictionaries[source]["average_rssi"] = 0
            source_dictionaries[source]["first_occurence"] = -1
            source_dictionaries[source]["last_occurence"] = 0
            source_dictionaries[source]["device"] = "empty"
            source_dictionaries[source]["sub_device"] = ""
            source_dictionaries[source]["group_id"] = 0


In [4]:
# Assign values for each row with either simple transformations or direct value usage
def fill_labelled_columns(packet_data: tuple, device: str, sub_device: str, group_id: int):

    row_data = {
        "time": packet_data.Time,
        "source": packet_data.Source,
        "destination": packet_data.Destination,
        "is_broadcast": assign_destination(packet_data.Destination),
        "length": packet_data.Length,
        "info": packet_data.Info,
        "rssi": packet_data.RSSI,
        "company_id": encode_company_id(packet_data.Company_ID, packet_data.Source, packet_data.Service_Data),
        "has_company_id": check_company_id_existence(encode_company_id(packet_data.Company_ID, packet_data.Source, packet_data.Service_Data)),
        "channel": packet_data.Channel_Index,
        "is_adv_channel": is_adv_channel(packet_data.Channel_Index),
        "device_name": check_null_value(packet_data.Device_Name),
        "uuid16": check_null_value(packet_data.UUID_16),
        "has_uuid16": check_entry_existence(packet_data.UUID_16),
        "len_uuid16": extract_length(packet_data.UUID_16),
        "uuid128": check_null_value(packet_data.UUID128),
        "has_uuid128": check_entry_existence(packet_data.UUID128),
        "data": check_null_value(packet_data.Data),
        "len_data": extract_length(packet_data.Data),
        "ad_type": check_null_value(packet_data.Type),
        "len_ad_type": extract_length(packet_data.Data),
        "service_data": check_null_value(packet_data.Service_Data),
        "len_service_data": extract_length(packet_data.Service_Data),
        "crc": check_null_value(packet_data.CRC),
        "labelled_device": assign_device(device, packet_data.Source),
        "sublabel_device": sub_device,
        "group_id": group_id,
        "group_label_bin": 1 if group_id != 0 else 0,
        "group_label_multi": group_id,  # 0=noise, 1=G1, 2=G2
        "time_start_end": extract_time(packet_data.packet_start_end),
        "delta_end_start": extract_time(packet_data.delta_end_start),
        "delta_start_start": extract_time(packet_data.delta_start_start)
    }

    # To ensure labeling quality, phone-based labels which do not a valid company_id are labeled empty
    # Valid company_ids include: ["Samsung Electronics Co. Ltd.", "Undefined", "Unknown"]
    if row_data["labelled_device"] in ['Phone'] and row_data["company_id"] not in phone_companies:
        row_data["labelled_device"] = "empty"
        row_data["sublabel_device"] = "empty"
        row_data["group_id"] = 0
        row_data["group_label_bin"] = 0
        row_data["group_label_multi"] = 0

    return row_data


In [5]:
# Main raw dataset input parsing method
def parse_dataframe(input_dataframe: pd.DataFrame):
    nan_discarded_counter = 0
    malformed_discarded_counter = 0
    smart_tag_list = []
    phone_list = []
    buds_list = []

    dict_mean_rssi = (
        pd.to_numeric(
            input_dataframe["RSSI"].astype(str).str.replace("dBm", "", regex=False).str.strip(),
            errors="coerce"
        )
        .groupby(input_dataframe["Source"])
        .mean()
        .to_dict()
    )

    for row in input_dataframe.itertuples(index=False):
        if str(row.Source) == "nan":
            nan_discarded_counter += 1
            continue

        source_dictionaries[row.Source]["count"] += 1

        if "Malformed Packet" in row.Info:
            source_dictionaries[row.Source]["malformed_count"] += 1
            malformed_discarded_counter += 1
            continue

        # Parse RSSI string "-50 dBm" -> int
        try:
            current_rssi = int(str(row.RSSI).replace("dBm","").strip())
        except:
            current_rssi = -100
        if source_dictionaries[row.Source]["first_occurence"] == -1:
            source_dictionaries[row.Source]["first_occurence"] = row.Time
        if row.Time > source_dictionaries[row.Source]["last_occurence"]:
            source_dictionaries[row.Source]["last_occurence"] = row.Time
        if current_rssi > source_dictionaries[row.Source]["lowest_rssi"]:
            source_dictionaries[row.Source]["lowest_rssi"] = current_rssi
        if current_rssi < source_dictionaries[row.Source]["highest_rssi"]:
            source_dictionaries[row.Source]["highest_rssi"] = current_rssi
        if source_dictionaries[row.Source]["average_rssi"] == 0:
            source_dictionaries[row.Source]["average_rssi"] = dict_mean_rssi[row.Source]
            
        # --- Priority 1: Whitelist / ground truth MAP for multi-group ---
        # This overrides heuristics and ensures G1 vs G2 separation even with identical packet content / same RSSI
        if row.Source in SOURCE_TO_GROUP:
            dev, sub, grp = SOURCE_TO_GROUP[row.Source]
            source_dictionaries[row.Source]["device"] = dev
            source_dictionaries[row.Source]["sub_device"] = sub
            source_dictionaries[row.Source]["group_id"] = grp
        else:
            # --- Fallback heuristics for non-target / noise and for completeness ---
            # Smart Tag fallback disabled for 2Groups hybrid dataset (no Smart Tags in experiment)
            # if row.Device_Name == "Smart Tag":
            #     source_dictionaries[row.Source]["device"] = "Smart Tag"
            #     source_dictionaries[row.Source]["sub_device"] = "Smart Tag"
            # if row.UUID_16 == "Samsung Electronics Co., Ltd.,Samsung Electronics Co., Ltd.":
            #     source_dictionaries[row.Source]["device"] = "Smart Tag"
            #     source_dictionaries[row.Source]["sub_device"] = "Smart Tag"
            pass
            # Phone via UUID heuristics
            if row.UUID_16 == "Google LLC" and (row.Length == 63 or row.Length == 42 or row.Length == 120): 
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Google"
            if row.Info == "ADV_EXT_IND" and row.Length == 39:
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Phone"
            if row.UUID_16 == "Samsung Electronics Co. Ltd." and row.Length == 57: 
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Phone"
            if row.Info == "SCAN_REQ" and row.Length == 38 and row.Destination in target_group_rsp_scrs:
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Scan"
            # Buds fallback: any Samsung ADV_SCAN_IND L63 with Samsung company is Buds (covers unknown Buds)
            # But whitelist already covers G1/G2 Buds, so here we only label remaining Buds as noise or generic
            if row.Info == "ADV_SCAN_IND" and row.Length == 63 and row.Company_ID == "Samsung Electronics Co. Ltd.":
                if source_dictionaries[row.Source]["device"] == "empty" and row.Source not in ALL_BUDS:
                    # keep as empty to avoid mislabeling interferers; or label generic Buds with group 0
                    pass
            # AirTag fallback: Apple ADV_IND L63 (all AirTags) - whitelist covers G1/G2, rest treated as generic AirTag noise
            if row.Company_ID == "Apple, Inc." and row.Info == "ADV_IND" and row.Length == 63:
                if source_dictionaries[row.Source]["device"] == "empty" and row.Source not in ALL_AIRTAGS:
                    # Keep as empty/noise for modeling; if you want to keep as noise AirTag, uncomment:
                    # source_dictionaries[row.Source]["device"] = "AirTag"
                    # source_dictionaries[row.Source]["sub_device"] = "AirTag"
                    # source_dictionaries[row.Source]["group_id"] = 0
                    pass

        # After assigning all values, create a transformed row for the final list of rows
        row_data = fill_labelled_columns(row, source_dictionaries[row.Source]["device"], source_dictionaries[row.Source]["sub_device"], source_dictionaries[row.Source]["group_id"])
        final_list.append(row_data)

    return nan_discarded_counter, malformed_discarded_counter


In [6]:
def find_empty_and_malformed_sources():
    only_malformed_sources = []
    empty_sources = []

    for key in source_dictionaries.keys():
        count = source_dictionaries[key]["count"]
        malformed_count = source_dictionaries[key]["malformed_count"]
        device = source_dictionaries[key]["device"]

        if (count - malformed_count) == 0:
            only_malformed_sources.append(key)
        
        if (count - malformed_count) != 0 and device == "empty":
            empty_sources.append(key)

    print(f"Only Malformed Sources: {len(only_malformed_sources)}")
    print(f"Empty Sources: {len(empty_sources)}")

    return only_malformed_sources, empty_sources

def reassign_sources():
    empty_occurrences = 0
    new_final_list = []

    for row_dict in final_list:
        if row_dict["labelled_device"] == "empty" and row_dict["company_id"] in phone_companies:
            current_source = row_dict["source"]
            # Only reassign if source has known device in dictionary (prevents overriding whitelist)
            if source_dictionaries[current_source]["device"] != "empty":
                row_dict["labelled_device"] = source_dictionaries[current_source]["device"]
                row_dict["sublabel_device"] = source_dictionaries[current_source]["sub_device"]
                row_dict["group_id"] = source_dictionaries[current_source]["group_id"]
                row_dict["group_label_bin"] = 1 if source_dictionaries[current_source]["group_id"] !=0 else 0
                row_dict["group_label_multi"] = source_dictionaries[current_source]["group_id"]

        if row_dict["labelled_device"] != "empty":
            new_final_list.append(row_dict)
        else:
            new_final_list.append(row_dict)
            empty_occurrences += 1

    return empty_occurrences, new_final_list


In [7]:
# Create the final dataframe based on the final_list rows
def create_labelled_dataframe(entry_list: list):
    new_df = pd.DataFrame(data=entry_list, columns=list((final_list[0].keys())))        
    return new_df

def write_new_dataframe(file_to_write: str, df_to_write: pd.DataFrame):
    df_to_write.to_csv(file_to_write, encoding='utf-8', index=False)
    print(f"New CSV written as: {file_to_write}")

def info_extractor(df_input: pd.DataFrame, wanted_label: str):
    seen_sources = {}
    for row in df_input.itertuples(index=False):
        if row.source not in seen_sources.keys() and row.sublabel_device == wanted_label:
            seen_sources[row.source] = {}
            seen_sources[row.source]["first"] = row.time
            seen_sources[row.source]["last"] = row.time
            seen_sources[row.source]["count"] = source_dictionaries[row.source]["count"]
            continue
        if row.source in seen_sources.keys() and row.sublabel_device == wanted_label:
            if seen_sources[row.source]["last"] < row.time:
                seen_sources[row.source]["last"] = row.time
    return seen_sources

def assign_majority_label(input_df: pd.DataFrame):
    # Compute the majority label per source for labelled_device, sublabel and group
    majority_labels = input_df.groupby('source')['labelled_device'].agg(lambda x: x.mode()[0])
    majority_sub = input_df.groupby('source')['sublabel_device'].agg(lambda x: x.mode()[0])
    majority_group = input_df.groupby('source')['group_id'].agg(lambda x: x.mode()[0])
    input_df['labelled_device'] = input_df['source'].map(majority_labels)
    input_df['sublabel_device'] = input_df['source'].map(majority_sub)
    input_df['group_id'] = input_df['source'].map(majority_group)
    input_df['group_label_bin'] = (input_df['group_id'] != 0).astype(int)
    input_df['group_label_multi'] = input_df['group_id']
    return input_df


In [8]:
##############################
#        CODE FIELD 0        #
##############################

# Global variables
final_list = []
source_dictionaries = {}
phone_companies = ["Samsung Electronics Co. Ltd.", "Undefined", "Unknown"]
undefined_list = []


In [9]:
##############################
#        CODE FIELD 1        #
##############################

# Setup all lists, dictionaries and the labeled dataframe
path_to_file = "../../data/2Groups_3Devices_FirstExperiment/2Groups_3Devices_FirstExperiment.csv"
raw_dataframe = read_data_to_df(path_to_file) 
all_sources = raw_dataframe["Source"].unique()
# SCAN_RSP targets: both Buds FE devices are the scan targets (include both groups)
target_group_rsp_scrs = raw_dataframe.loc[(raw_dataframe["Info"] == "SCAN_RSP") & (raw_dataframe["Device_Name"] == "Buds FE"), "Source"].unique()
# Also include if Device_Name empty but Data indicates Buds: fallback
if len(target_group_rsp_scrs)==0:
    # Fallback: use whitelist Buds as rsp sources
    target_group_rsp_scrs = BUDS_G1 + BUDS_G2
print(f"target_group_rsp_scrs (Buds AdvA for Phone SCAN_REQ detection): {list(target_group_rsp_scrs)}")
initialize_source_dictionaries(all_sources)
nan_counter, malformed_counter = parse_dataframe(raw_dataframe)
malformed_list, empty_list = find_empty_and_malformed_sources()
empty_counter, filtered_list = reassign_sources()
labelled_dataframe = create_labelled_dataframe(filtered_list)
relevant_sources = labelled_dataframe["source"].unique()
print(f"Sources total: {len(all_sources)}, relevant labelled rows: {len(labelled_dataframe)}")


target_group_rsp_scrs (Buds AdvA for Phone SCAN_REQ detection): ['f0:c8:c1:88:8c:53', 'dc:b9:92:eb:f6:5d', 'f0:c8:c1:8a:8c:53', 'f0:c8:c1:b8:8c:53']
Only Malformed Sources: 627
Empty Sources: 700
Sources total: 1349, relevant labelled rows: 66618


In [10]:
##############################
#        CODE FIELD 2        #
##############################

# For multi-group experiment at same distance, RSSI must NOT be used to separate G1 vs G2.
# We only use count-based filtering to remove spurious interferers, and we keep all whitelisted sources regardless of RSSI.
# If you later want RSSI-based noise filtering, adjust per device type, but keep it permissive.

# Keep all whitelisted sources even if count low (G2 phone has 79 for first rotation)
whitelisted_sources = set(SOURCE_TO_GROUP.keys())

# Optional loose count filter for non-whitelisted sources: count>30 keeps only persistent devices, others become empty anyway
# We skip strict RSSI limits here to remain distance-agnostic.

# Example of how you could apply permissive filtering if needed (commented out for now):
# rssi_limits_per_dataset = {"multi_group": {"Phone Scan": -80, "Buds": -90, "AirTag": -90}}
# ... apply only to non-whitelisted sources

# Ensure whitelisted sources are never filtered out
for src in whitelisted_sources:
    # Ensure they are marked with correct device/group even if they had few packets
    if src in source_dictionaries:
        pass

print("--------------------------------------------------")
print("FINAL DATAFRAME (before majority vote)")
print(f"Length Raw Dataframe: {len(raw_dataframe)}")
print(f"Length Final Dataframe (incl. empty): {len(labelled_dataframe)}")
print(f"Removed: {malformed_counter} (malformed), {nan_counter} (NaN), Diff: {len(raw_dataframe)-len(labelled_dataframe)}")
print(f"Whitelisted target sources preserved: {len(whitelisted_sources)}")
# Show distribution before majority
print(labelled_dataframe.groupby("labelled_device").agg(packets=("labelled_device", "size"), sources=("source", "nunique")).sort_values(by="packets", ascending=False))

# Assign the majority label to impure sources (e.g., a source may have 3000 'Phone' and 1 'empty')
labelled_dataframe = assign_majority_label(labelled_dataframe)


--------------------------------------------------
FINAL DATAFRAME (before majority vote)
Length Raw Dataframe: 67808
Length Final Dataframe (incl. empty): 66618
Removed: 1041 (malformed), 149 (NaN), Diff: 1190
Whitelisted target sources preserved: 11
                 packets  sources
labelled_device                  
empty              32043      701
Buds               22950        2
Phone               8495       17
AirTag              3130        2


In [11]:
##############################
#        CODE FIELD 3        #
##############################

print("\n--- After majority vote ---")
print(labelled_dataframe.groupby("labelled_device").agg(packets=("labelled_device", "size"), sources=("source", "nunique")).sort_values(by="packets", ascending=False))
print("\nGroup distribution:")
print(labelled_dataframe.groupby("group_id").agg(packets=("group_id", "size"), sources=("source", "nunique")).sort_values(by="group_id"))
print("\nDetailed per device + group:")
print(labelled_dataframe.groupby(["labelled_device","group_id"]).agg(packets=("labelled_device", "size"), sources=("source", "nunique")))
# Check if there are still impure sources which have multiple labels
multi_label_sources = labelled_dataframe.groupby('source')['labelled_device'].nunique()
multi_label_sources = multi_label_sources[multi_label_sources > 1].index
print(f"\nSources with multiple labelled_device after majority: {len(multi_label_sources)} (should be 0)")
for source in multi_label_sources:
    print("\n", labelled_dataframe[labelled_dataframe["source"] == source]["labelled_device"].value_counts())

multi_group_sources = labelled_dataframe.groupby('source')['group_id'].nunique()
multi_group_sources = multi_group_sources[multi_group_sources > 1].index
print(f"\nSources with multiple group_id after majority: {len(multi_group_sources)} (should be 0)")



--- After majority vote ---
                 packets  sources
labelled_device                  
empty              35249      701
Buds               22950        2
Phone               5289       16
AirTag              3130        2

Group distribution:
          packets  sources
group_id                  
0           36575      710
1           15891        5
2           14152        6

Detailed per device + group:
                          packets  sources
labelled_device group_id                  
AirTag          1            1602        1
                2            1528        1
Buds            1           11702        1
                2           11248        1
Phone           0            1326        9
                1            2587        3
                2            1376        4
empty           0           35249      701

Sources with multiple labelled_device after majority: 0 (should be 0)

Sources with multiple group_id after majority: 0 (should be 0)


In [12]:
def group_by_src(df):
    # Simple robust version - converts RSSI to numeric first
    df = df.copy()
    df["rssi_num"] = pd.to_numeric(df["rssi"].astype(str).str.replace("dBm","",regex=False).str.strip(), errors="coerce")
    # Use simple agg without dict lambda that can fail
    result = (
    df.groupby("source")
      .agg(
          count=("rssi_num", "count"),
          RSSI_min=("rssi_num", "min"),
          RSSI_max=("rssi_num", "max"),
          RSSI_avg=("rssi_num", "mean"),
          label=("labelled_device", lambda x: ", ".join(x.unique())),
          group=("group_id", lambda x: ", ".join(map(str, x.unique())))
      )
      .reset_index()                
      .sort_values(by="count", ascending=False)
      .reset_index(drop=True)
    )
    return result


In [13]:
##############################
#        CODE FIELD 4        #
##############################

# For this multi-group dataset we keep empty/noise rows for binary and 3-class training
# If you have an isolated capture and want to remove noise, uncomment next block
# to_drop = (labelled_dataframe["labelled_device"] == "empty")
# labelled_dataframe = labelled_dataframe.drop(labelled_dataframe[to_drop].index)

# Safe preview without complex agg that fails on string RSSI
try:
    print(group_by_src(labelled_dataframe[labelled_dataframe["labelled_device"] != "empty"]).head(20).to_string())
except Exception as e:
    print(f"group_by_src preview failed: {e}")
    # Fallback simple preview
    print(labelled_dataframe[labelled_dataframe["labelled_device"] != "empty"].groupby("source").size().head(20))

try:
    print("\nNoise sources preview:")
    print(group_by_src(labelled_dataframe[labelled_dataframe["labelled_device"] == "empty"]).head(10).to_string())
except Exception as e:
    print(f"noise preview failed: {e}")

# save labelled dataframe as csv
# Use forward compatible naming for modeling
new_file_name = "2Groups_3Devices_FirstExperiment_labeled"
labelled_dataframe.to_csv(f"../../data/2Groups_3Devices_FirstExperiment/{new_file_name}.csv", index=False)
print(f"\nNew CSV written as: ../../data/2Groups_3Devices_FirstExperiment/{new_file_name}.csv")
print(f"Columns: {list(labelled_dataframe.columns)}")
print(f"For modeling: use group_label_bin for binary (Target vs Noise) or group_label_multi for 3-class (0 noise,1 G1,2 G2)")


               source  count  RSSI_min  RSSI_max   RSSI_avg   label group
0   dc:b9:92:eb:f6:5d  11702       -91       -40 -51.309862    Buds     1
1   f0:c8:c1:88:8c:53  11248      -106       -50 -64.895359    Buds     2
2   fb:28:4e:b6:e1:c1   1602       -24       -16 -17.720974  AirTag     1
3   d2:5f:07:c2:32:c0   1528       -67       -36 -51.863874  AirTag     2
4   59:2a:92:26:ae:ac    947       -41       -17 -20.921859   Phone     1
5   71:9c:05:bf:d0:e9    864       -23       -17 -19.563657   Phone     1
6   48:1a:0a:8d:24:91    776       -28       -20 -24.331186   Phone     1
7   7c:96:71:24:45:f6    718       -91       -64 -79.029248   Phone     0
8   7a:e4:ca:23:f9:3e    563       -88       -65 -72.676732   Phone     0
9   65:cd:d7:af:54:cb    516       -79       -48 -56.662791   Phone     2
10  7b:1f:3c:ec:d0:6c    500       -77       -49 -60.684000   Phone     2
11  49:f7:63:04:de:64    281       -57       -47 -52.814947   Phone     2
12  62:ad:82:a0:76:e6     79       -70